# Theoretical Optimal Compression for Low-Feature Cases

Instead of training models, we can reason about what the **optimal** hidden-space arrangement looks like for small n, m. Two constraints lead to two different geometries:

1. **Equal-magnitude constraint**: All features must map to vectors of the same norm in latent space → optimal packing is a regular n-gon (or its higher-dimensional analogue)
2. **Linear trajectory + variable gradient**: Feature trajectories must follow straight lines (positive homogeneity) but the decoder can have nonlinear read-out → "onion features" where a feature speeds past other features' decoder regions and slows down in its own

Working in m=2 throughout for visualization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
from scipy.optimize import minimize
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## Part 1: Equal-Magnitude Constraint → Optimal Angular Packing

### Setup

Suppose we have n sparse non-negative features and a bottleneck of dimension m=2. The encoder maps each feature direction $e_i$ to some latent vector $w_i \in \mathbb{R}^2$.

**Constraint**: All features have equal importance, so $\|w_i\| = r$ for all $i$ (same magnitude).

**Decoder**: Given latent code $z$, the decoder reconstructs by finding which feature(s) $z$ is closest to, weighted by projection. For a linear decoder with tied weights, the reconstruction of feature $i$ is $\hat{x}_i = w_i^T z$.

**Interference**: When feature $i$ is active (input $= t \cdot e_i$), the encoding is $z = t \cdot w_i$. The decoder reconstructs feature $j \neq i$ as $\hat{x}_j = t \cdot w_j^T w_i = t r^2 \cos\theta_{ij}$. This is **interference** — the spurious reconstruction of inactive features.

**Optimal arrangement**: Minimize total interference $\sum_{i \neq j} (w_j^T w_i)^2$, which under equal norms means minimize $\sum_{i < j} \cos^2\theta_{ij}$.

This is a variant of the **Tammes problem** (packing points on a sphere to maximize minimum distance).

### Analytical solutions for m=2

In 2D, the feature vectors $w_i$ are points on a circle. With the non-negativity constraint on inputs (features are non-negative), the encoder only needs to handle the positive orthant. Key cases:

| n | m=2 | Optimal arrangement | Min angle between adjacent | Interference per pair |
|---|-----|--------------------|--------------------------|-----------------------|
| 2 | 2   | Orthogonal (90°)   | 90° | 0 |
| 3 | 2   | 120° apart (or use non-negativity: 60° sector with 3 vectors) | 60° | $r^4/4$ |
| 4 | 2   | 45° apart or n-gon (90° spacing) | 45° | higher |
| n | 2   | Uniform angular spacing: $\theta_{ij} = \pi(i-j)/n$ or pack into half-plane | $\pi/n$ | increases with n |

In [ ]:
def compute_interference_loss(angles, S=0.95):
    """Compute expected MSE from interference between features.
    
    For sparse features with sparsity S, feature i is active with prob (1-S).
    When feature i is active with magnitude t ~ U[0,1], the spurious reconstruction
    of feature j is: x_hat_j = t * r^2 * cos(theta_ij)
    
    But with ReLU output, negative reconstructions are clipped to 0.
    So interference only happens when cos(theta_ij) > 0, i.e. angle < 90°.
    
    Expected squared interference from feature i onto j (when i active, j inactive):
    E[t^2] * r^4 * max(0, cos(theta_ij))^2 = (1/3) * r^4 * max(0, cos(theta_ij))^2
    
    Args:
        angles: array of angles in radians for each feature vector
        S: sparsity (prob of being zero)
    Returns:
        total expected MSE (assuming r=1 for simplicity)
    """
    n = len(angles)
    # Feature vectors on unit circle
    vectors = np.stack([np.cos(angles), np.sin(angles)], axis=1)
    
    total_interference = 0.0
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            cos_ij = np.dot(vectors[i], vectors[j])
            # ReLU clips negative reconstructions
            interference = max(0, cos_ij) ** 2
            # Prob(i active, j inactive) = (1-S) * S
            total_interference += (1 - S) * S * (1/3) * interference
    
    return total_interference


def compute_representation_loss(angles, S=0.95):
    """Compute expected MSE from imperfect self-reconstruction.
    
    With tied weights and unit-norm vectors, self-reconstruction of feature i is:
    x_hat_i = t * w_i^T w_i = t * r^2 = t (if r=1)
    
    So self-reconstruction is perfect! The loss comes only from interference.
    But if r != 1, there's a bias: x_hat_i = t * r^2, and optimal r = 1.
    
    For non-tied weights, the story is different.
    """
    return 0.0  # Perfect self-reconstruction with r=1, tied weights


def optimal_ngon_angles(n, sector=np.pi):
    """Regular n-gon arrangement within a sector.
    
    With non-negative features + ReLU output, we only need the first quadrant
    or first half-plane. Using sector=pi/2 for first quadrant.
    """
    if n == 1:
        return np.array([sector / 2])
    return np.linspace(0, sector, n, endpoint=True)


# Visualize optimal arrangements for n=2,3,4,5,6
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, n in enumerate([2, 3, 4, 5, 6, 8]):
    ax = axes[idx]
    
    # Try different sector sizes
    # With non-negative features, all angles in [0, pi/2] to stay in first quadrant
    # But the decoder has ReLU, so negative outputs are fine (clipped to 0)
    # So we can use full circle, but features pointing in opposite directions
    # are wasteful (ReLU kills the negative reconstruction)
    # Optimal: spread across [0, pi] (half-plane) for non-negative features
    
    sector = np.pi  # half-plane
    angles = optimal_ngon_angles(n, sector=sector)
    
    # Plot unit circle
    theta = np.linspace(0, 2*np.pi, 200)
    ax.plot(np.cos(theta), np.sin(theta), 'k-', alpha=0.2, linewidth=0.5)
    
    # Plot feature vectors
    colors = plt.cm.tab10(np.linspace(0, 1, n))
    for i in range(n):
        wx, wy = np.cos(angles[i]), np.sin(angles[i])
        ax.arrow(0, 0, wx*0.9, wy*0.9, head_width=0.06, head_length=0.05,
                fc=colors[i], ec=colors[i], linewidth=2)
        ax.text(wx*1.15, wy*1.15, f'$w_{i}$', fontsize=10, ha='center', va='center',
               color=colors[i], fontweight='bold')
    
    # Shade the sector
    sector_theta = np.linspace(0, sector, 100)
    sector_x = np.concatenate([[0], np.cos(sector_theta), [0]])
    sector_y = np.concatenate([[0], np.sin(sector_theta), [0]])
    ax.fill(sector_x, sector_y, alpha=0.05, color='blue')
    
    # Compute interference
    loss = compute_interference_loss(angles, S=0.95)
    min_angle = np.min(np.diff(np.sort(angles))) * 180 / np.pi if n > 1 else 180
    
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_title(f'n={n}, min angle={min_angle:.0f}°\nInterference MSE={loss:.4f}')

plt.suptitle('Optimal Equal-Magnitude Packing in m=2 (half-plane sector)', fontsize=14)
plt.tight_layout()
plt.show()

### How fast does interference grow with n?

For $n$ features equally spaced in a half-plane $[0, \pi]$, the angle between adjacent features is $\pi/(n-1)$. Total interference scales roughly as $\sum_{i<j} \cos^2(\pi|i-j|/(n-1))$.

In [ ]:
n_range = np.arange(2, 25)
interference_by_n = []
min_angles = []

for n in n_range:
    angles = optimal_ngon_angles(n, sector=np.pi)
    loss = compute_interference_loss(angles, S=0.95)
    interference_by_n.append(loss)
    if n > 1:
        min_angles.append(np.min(np.diff(np.sort(angles))) * 180 / np.pi)
    else:
        min_angles.append(180)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
ax.plot(n_range, interference_by_n, 'bo-', markersize=5)
ax.set_xlabel('n (features)')
ax.set_ylabel('Expected interference MSE')
ax.set_title('Total Interference vs Feature Count\n(m=2, S=0.95, half-plane packing)')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(n_range, min_angles, 'ro-', markersize=5)
ax.set_xlabel('n (features)')
ax.set_ylabel('Minimum pairwise angle (degrees)')
ax.set_title('Angular Separation vs Feature Count')
ax.axhline(90, color='green', linestyle='--', alpha=0.5, label='90° (orthogonal)')
ax.legend()
ax.grid(True, alpha=0.3)

# Per-feature interference
ax = axes[2]
per_feature = np.array(interference_by_n) / n_range
ax.plot(n_range, per_feature, 'go-', markersize=5)
ax.set_xlabel('n (features)')
ax.set_ylabel('Interference MSE per feature')
ax.set_title('Per-Feature Interference')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key transition points:")
for n, angle, loss in zip(n_range, min_angles, interference_by_n):
    if n <= 6 or n in [8, 12, 16, 24]:
        print(f"  n={n:2d}: min_angle={angle:5.1f}°, interference={loss:.5f}")

### When does it become worth it to drop features?

At some point, the interference from packing too many features is worse than just ignoring the least important feature. With importance weighting $I_i = \text{decay}^i$, the cost of dropping feature $i$ is proportional to $I_i \cdot (1-S)$. We can compare this to the interference reduction from removing it.

In [ ]:
def compute_optimal_representation(n_total, m=2, S=0.95, importance_decay=0.7):
    """For n_total features, find the optimal number to represent.
    
    Returns dict with:
    - n_represented: optimal number of features to encode
    - total_loss: MSE for optimal representation
    - losses_by_k: MSE for each possible k in [1, n_total]
    """
    importance = np.array([importance_decay ** i for i in range(n_total)])
    
    losses_by_k = []
    for k in range(1, n_total + 1):
        # Represent top-k features, drop the rest
        angles = optimal_ngon_angles(k, sector=np.pi)
        
        # Interference between represented features
        vectors = np.stack([np.cos(angles), np.sin(angles)], axis=1)
        interference = 0.0
        for i in range(k):
            for j in range(k):
                if i == j:
                    continue
                cos_ij = np.dot(vectors[i], vectors[j])
                # Weighted by importance of the interfered-upon feature
                interference += importance[j] * (1 - S) * S * (1/3) * max(0, cos_ij) ** 2
        
        # Cost of dropping features k through n_total-1
        # If feature i is dropped, its expected squared error is I_i * (1-S) * E[t^2] = I_i * (1-S) * 1/3
        drop_cost = sum(importance[i] * (1 - S) * (1/3) for i in range(k, n_total))
        
        total = interference + drop_cost
        losses_by_k.append(total)
    
    best_k = np.argmin(losses_by_k) + 1
    return {
        'n_represented': best_k,
        'total_loss': losses_by_k[best_k - 1],
        'losses_by_k': losses_by_k
    }


fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, decay in zip(axes, [0.9, 0.7, 0.5]):
    for n_total in [4, 6, 8, 12, 16]:
        result = compute_optimal_representation(n_total, S=0.95, importance_decay=decay)
        k_range = np.arange(1, n_total + 1)
        ax.plot(k_range, result['losses_by_k'], 'o-', markersize=4,
               label=f'n={n_total} (opt={result["n_represented"]})')
    
    ax.set_xlabel('k (features represented)')
    ax.set_ylabel('Total expected MSE')
    ax.set_title(f'Optimal Representation Count\n(decay={decay}, S=0.95, m=2)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Optimal packing with unequal magnitudes

When features have unequal importance, the optimal arrangement is no longer a regular n-gon. More important features should get **larger norms** (more representation capacity) while less important features get smaller norms. The angles should also adjust — more important features should be farther from other features.

Let's optimize both angles and magnitudes jointly.

In [ ]:
def compute_weighted_loss(params, n, S=0.95, importance_decay=0.7):
    """Total loss with variable angles AND magnitudes.
    
    params[:n] = angles
    params[n:] = log-magnitudes (use log to keep positive)
    """
    angles = params[:n]
    log_mags = params[n:]
    mags = np.exp(log_mags)
    
    importance = np.array([importance_decay ** i for i in range(n)])
    
    vectors = np.stack([mags * np.cos(angles), mags * np.sin(angles)], axis=1)
    
    total_loss = 0.0
    for i in range(n):
        # Self-reconstruction: x_hat_i = t * w_i^T w_i = t * ||w_i||^2
        # Error: (t - t*||w_i||^2)^2 = t^2 * (1 - ||w_i||^2)^2
        # E[t^2] = 1/3 for U[0,1]
        self_err = importance[i] * (1 - S) * (1/3) * (1 - mags[i]**2)**2
        total_loss += self_err
        
        # Interference from feature i onto feature j
        for j in range(n):
            if i == j:
                continue
            # When i is active, j's reconstruction = t * w_j^T w_i
            dot_ij = np.dot(vectors[i], vectors[j])
            interference = max(0, dot_ij) ** 2
            total_loss += importance[j] * (1 - S) * S * (1/3) * interference
    
    return total_loss


def optimize_arrangement(n, S=0.95, importance_decay=0.7, n_restarts=20):
    """Find optimal angles and magnitudes by numerical optimization."""
    best_result = None
    best_loss = np.inf
    
    for _ in range(n_restarts):
        # Random initial angles in [0, pi], magnitudes near 1
        x0 = np.concatenate([
            np.sort(np.random.uniform(0, np.pi, n)),
            np.random.normal(0, 0.3, n)  # log-magnitudes near 0 → mags near 1
        ])
        
        result = minimize(
            compute_weighted_loss, x0,
            args=(n, S, importance_decay),
            method='L-BFGS-B'
        )
        
        if result.fun < best_loss:
            best_loss = result.fun
            best_result = result
    
    angles = best_result.x[:n]
    mags = np.exp(best_result.x[n:])
    return angles, mags, best_loss


# Compare equal-mag (n-gon) vs optimized for different n
fig, axes = plt.subplots(2, 3, figsize=(16, 11))

for idx, n in enumerate([3, 4, 5, 6, 8, 10]):
    ax = axes[idx // 3, idx % 3]
    
    # Optimized arrangement
    opt_angles, opt_mags, opt_loss = optimize_arrangement(n, importance_decay=0.7)
    
    # Equal-magnitude baseline
    eq_angles = optimal_ngon_angles(n, sector=np.pi)
    eq_loss = compute_weighted_loss(
        np.concatenate([eq_angles, np.zeros(n)]),  # log(1) = 0
        n, importance_decay=0.7
    )
    
    importance = np.array([0.7 ** i for i in range(n)])
    
    # Plot unit circle
    theta = np.linspace(0, 2*np.pi, 200)
    ax.plot(np.cos(theta), np.sin(theta), 'k-', alpha=0.15, linewidth=0.5)
    
    # Plot equal-magnitude vectors (thin, gray)
    for i in range(n):
        wx, wy = np.cos(eq_angles[i]), np.sin(eq_angles[i])
        ax.arrow(0, 0, wx*0.85, wy*0.85, head_width=0.04, head_length=0.03,
                fc='gray', ec='gray', alpha=0.4, linewidth=1)
    
    # Plot optimized vectors (thick, colored by importance)
    colors = plt.cm.YlOrRd(importance / importance.max())
    max_mag = max(opt_mags)
    for i in range(n):
        r = opt_mags[i] / max_mag  # normalize for display
        wx, wy = r * np.cos(opt_angles[i]), r * np.sin(opt_angles[i])
        ax.arrow(0, 0, wx*0.85, wy*0.85, head_width=0.05, head_length=0.04,
                fc=colors[i], ec=colors[i], linewidth=2)
        ax.text(wx*1.1, wy*1.1, f'{i}', fontsize=8, ha='center', va='center',
               fontweight='bold')
    
    improvement = (eq_loss - opt_loss) / eq_loss * 100
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    ax.set_title(f'n={n}: equal={eq_loss:.4f}, opt={opt_loss:.4f}\n({improvement:.1f}% improvement)')

plt.suptitle('Equal-Magnitude (gray) vs Optimized (colored) Arrangements\n(color = importance, size = magnitude)',
            fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Inspect the optimized arrangements more carefully
print("Optimized arrangements (importance_decay=0.7, S=0.95):")
print("="*70)

for n in [3, 4, 6, 8]:
    opt_angles, opt_mags, opt_loss = optimize_arrangement(n, importance_decay=0.7)
    
    # Sort by angle for readability
    sort_idx = np.argsort(opt_angles)
    
    print(f"\nn={n}: total MSE = {opt_loss:.6f}")
    print(f"  {'Feature':>8} {'Importance':>10} {'Angle(°)':>10} {'Magnitude':>10}")
    for i in range(n):
        imp = 0.7 ** i
        print(f"  {i:>8} {imp:>10.4f} {opt_angles[i]*180/np.pi:>10.1f} {opt_mags[i]:>10.4f}")
    
    # Check: do more important features get larger magnitudes?
    rank_corr = np.corrcoef(np.arange(n), opt_mags)[0, 1]
    print(f"  Correlation(feature_rank, magnitude): {rank_corr:.3f}")
    print(f"  (Negative = more important features get larger magnitudes)")

## Part 2: Linear Trajectories with Variable Speed → Onion Features

### The key idea

With bias-free ReLU networks, $z(t \cdot e_i) = t \cdot z(e_i)$ — trajectories are perfectly straight and uniform-speed. The feature moves at constant speed along its ray.

But what if we allow **biases** in the encoder? Then $z(t \cdot e_i)$ can be nonlinear in $t$. The trajectory can:
- Still follow a straight line in direction (or nearly so)
- But **change speed**: move quickly through some regions and slowly through others

This enables a strategy:
1. Multiple features share the **same direction** in latent space
2. The decoder assigns different **radial shells** to different features
3. Each feature's trajectory **speeds past** other features' shells and **slows down** in its own shell

This is exactly the onion feature concept. Let's build it from scratch.

In [ ]:
def sigmoid(x, center, steepness):
    """Shifted sigmoid."""
    return 1 / (1 + np.exp(-steepness * (x - center)))


def construct_onion_trajectory(t_vals, shell_radius, shell_width=0.3, steepness=10):
    """Construct a trajectory that saturates at a given shell radius.
    
    The trajectory rises quickly to shell_radius and then plateaus.
    This means it 'speeds past' lower shells and 'parks' in its shell.
    
    Args:
        t_vals: input magnitudes
        shell_radius: radius at which this feature saturates
        shell_width: width of transition region
        steepness: how sharp the transition is
    """
    # Sigmoid that maps [0, inf) → [0, shell_radius]
    return shell_radius * sigmoid(t_vals, 0.3, steepness)


# Demonstrate the concept with 4 features sharing one direction
t_vals = np.linspace(0, 2, 500)
shell_radii = [3.0, 2.0, 1.2, 0.5]  # Decreasing by importance

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Trajectory radius vs input magnitude
ax = axes[0]
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
for i, (r, c) in enumerate(zip(shell_radii, colors)):
    traj_r = construct_onion_trajectory(t_vals, r, steepness=8)
    ax.plot(t_vals, traj_r, color=c, linewidth=2.5, label=f'Feature {i} (shell r={r})')

# Draw shell boundaries
for i, r in enumerate(shell_radii):
    ax.axhline(r, color=colors[i], linestyle=':', alpha=0.3)

ax.set_xlabel('Input magnitude t', fontsize=12)
ax.set_ylabel('||z(t·eᵢ)||', fontsize=12)
ax.set_title('(a) Onion: Radius vs Input Magnitude\nEach feature saturates at its shell', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# (b) Speed (d||z||/dt) vs input magnitude
ax = axes[1]
for i, (r, c) in enumerate(zip(shell_radii, colors)):
    traj_r = construct_onion_trajectory(t_vals, r, steepness=8)
    speed = np.gradient(traj_r, t_vals)
    ax.plot(t_vals, speed, color=c, linewidth=2, label=f'Feature {i}')

ax.set_xlabel('Input magnitude t', fontsize=12)
ax.set_ylabel('d||z||/dt (speed)', fontsize=12)
ax.set_title('(b) Trajectory Speed\nHigh speed = passing through, low = parked', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# (c) 2D visualization with shared direction
ax = axes[2]
direction = np.array([np.cos(np.pi/4), np.sin(np.pi/4)])  # 45° direction

for i, (r, c) in enumerate(zip(shell_radii, colors)):
    traj_r = construct_onion_trajectory(t_vals, r, steepness=8)
    traj_2d = np.outer(traj_r, direction)  # (n_points, 2)
    ax.plot(traj_2d[:, 0], traj_2d[:, 1], color=c, linewidth=2.5,
           label=f'Feature {i}')
    # Mark saturation point
    ax.scatter(r*direction[0], r*direction[1], color=c, s=80, zorder=5,
              edgecolors='black', linewidths=1)

# Draw concentric shells
for i, (r, c) in enumerate(zip(shell_radii, colors)):
    circle = plt.Circle((0, 0), r, fill=False, color=c, linestyle='--', alpha=0.3)
    ax.add_patch(circle)

ax.set_xlabel('z₀', fontsize=12)
ax.set_ylabel('z₁', fontsize=12)
ax.set_title('(c) Trajectories in 2D Latent Space\n(all share same direction)', fontsize=11)
ax.set_xlim(-0.5, 3.5)
ax.set_ylim(-0.5, 3.5)
ax.set_aspect('equal')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Why onion is better than angular packing (at high compression)

**Angular packing** uses n distinct directions in m=2. The minimum angle shrinks as $\pi/(n-1)$, so interference grows quadratically.

**Onion packing** uses 1 direction but n distinct radii. There's no angular interference at all! The cost is that the decoder must be nonlinear (to read different features at different radii), and the encoder needs biases (to create the saturating trajectories).

**Key tradeoff**:
- Angular packing: linear decoder works, but interference grows with n
- Onion packing: zero angular interference, but needs nonlinear decoder and more careful magnitude discrimination

In practice, you'd use a **hybrid**: a few directions, each with a few radial shells. Let's model this.

In [ ]:
def compute_onion_decoder_loss(shell_radii, shell_boundaries, direction, S=0.95, importance_decay=0.7):
    """Compute MSE for an onion encoding along a single direction.
    
    The decoder partitions the ray into radial zones:
    - Between boundary[i-1] and boundary[i], the decoder outputs feature i
    - The decoded magnitude is proportional to how far into the zone the code is
    
    Key error sources:
    1. Quantization: feature magnitude maps to finite-precision radial position
    2. Shell confusion: when close to a boundary, decoder may misclassify
    """
    n = len(shell_radii)
    importance = np.array([importance_decay ** i for i in range(n)])
    
    # For each feature, compute reconstruction error
    # Feature i is encoded at radius r_i(t) ≈ shell_radii[i] * sigmoid(t)
    # The decoder reads: if r is in [boundary[i], boundary[i+1]], output feature i
    # with magnitude proportional to (r - boundary[i]) / (boundary[i+1] - boundary[i])
    
    total_mse = 0.0
    n_samples = 1000
    t_samples = np.random.uniform(0, 1, n_samples)
    
    for i in range(n):
        # Feature i active with magnitude t
        # Encoded at radius: shell_radii[i] * sigmoid(t, ...)
        encoded_r = shell_radii[i] * sigmoid(t_samples, 0.3, 8)
        
        # Decoder reads this radius: which zone does it fall in?
        # And what magnitude does it reconstruct?
        for t_idx, (t, r) in enumerate(zip(t_samples, encoded_r)):
            # Find which zone r falls in
            decoded_feature = -1
            decoded_mag = 0.0
            for j in range(n):
                lo = shell_boundaries[j]
                hi = shell_boundaries[j + 1]
                if lo <= r < hi:
                    decoded_feature = j
                    decoded_mag = (r - lo) / (hi - lo)  # Normalized position in zone
                    break
            
            if decoded_feature == -1:
                # Above all boundaries → assign to outermost feature
                decoded_feature = 0
                decoded_mag = 1.0
            
            # Reconstruction error
            if decoded_feature == i:
                # Correct feature, but magnitude may be wrong
                total_mse += importance[i] * (t - decoded_mag) ** 2
            else:
                # Wrong feature entirely
                total_mse += importance[i] * t ** 2  # Missed feature i
                total_mse += importance[decoded_feature] * decoded_mag ** 2  # Spurious feature j
    
    return total_mse / n_samples * (1 - S)  # Weight by probability of being active


# This is getting complicated with the decoder zones — let's think about it differently.
# The real question is: how many features can share a direction?

print("Let's think about this more carefully...")

### A cleaner model: capacity per direction

Rather than modeling the full decoder, let's think about **how many features a single direction can support**.

**Linear encoding** (no biases): Each direction supports exactly 1 feature. The magnitude along that direction linearly encodes the feature's value. If two features share a direction, their magnitudes are confounded.

**Onion encoding** (with biases): Each direction can support multiple features via radial shells. But there are limits:

1. **Shell separation**: Adjacent shells need sufficient gap for the decoder to distinguish them. If feature values are in $[0, 1]$ and there are $k$ shells per direction, each shell gets $\sim 1/k$ of the radial range → magnitude precision scales as $1/k$.

2. **Sparsity helps**: With sparsity $S$, the probability of two features on the same direction being simultaneously active is $(1-S)^2$. At $S = 0.95$, this is only 0.25%. So shell confusion matters mainly for the "which feature is it" question, not for co-activation.

3. **Magnitude quantization**: With $k$ shells per direction, each feature's magnitude has $\sim R/k$ radial range, where $R$ is the total radius. The reconstruction error from quantization scales as $\sim 1/k^2$.

**Effective capacity of onion encoding**: $k$ features per direction, each with $1/k$ magnitude resolution → **direction-sharing is free for identity, costly for magnitude precision**.

In [ ]:
def angular_packing_mse(n, m=2, S=0.95):
    """MSE for angular-only packing (equal magnitude, no onion).
    
    n features uniformly distributed in m=2 half-plane.
    Interference: sum of cos^2(angle_ij) for angle < 90°.
    """
    if n <= m:
        return 0.0  # Can be orthogonal
    
    angles = optimal_ngon_angles(n, sector=np.pi)
    return compute_interference_loss(angles, S=S)


def onion_packing_mse(n, n_directions, m=2, S=0.95):
    """MSE for hybrid angular + onion packing.
    
    n features distributed across n_directions directions,
    with n/n_directions features per direction (onion shells).
    
    Error sources:
    1. Angular interference between directions
    2. Radial quantization within each direction (1/k precision)
    """
    if n_directions > n:
        n_directions = n
    
    k = n / n_directions  # Features per direction
    
    # Angular interference (between the n_directions directions)
    if n_directions <= m:
        angular_loss = 0.0
    else:
        dir_angles = optimal_ngon_angles(n_directions, sector=np.pi)
        angular_loss = compute_interference_loss(dir_angles, S=S)
        # Scale by number of features (each direction has k features)
        angular_loss *= k  # More features per direction = more interference
    
    # Radial quantization error
    # Each feature gets 1/k of the radial range
    # Magnitude reconstruction error ~ (1/k)^2 / 12 (uniform quantization)
    # But with sigmoid saturation, it's more like the derivative at the shell boundary
    quantization_loss = n * (1 - S) * (1/3) * (1 / (12 * k**2)) if k > 1 else 0
    
    # Co-activation confusion (two features on same direction both active)
    # Probability: (1-S)^2 per pair on same direction
    # Each direction has k*(k-1)/2 pairs
    coactivation_loss = n_directions * (k * (k-1) / 2) * (1-S)**2 * (1/3)
    
    return angular_loss + quantization_loss + coactivation_loss


# Compare strategies across feature counts
n_range = np.arange(2, 20)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for S, ax in zip([0.95, 0.99], axes):
    pure_angular = [angular_packing_mse(n, S=S) for n in n_range]
    
    # Try different numbers of directions for onion
    for n_dir_label, n_dir_func in [
        ('1 dir (pure onion)', lambda n: 1),
        ('2 dirs', lambda n: 2),
        ('sqrt(n) dirs', lambda n: max(1, int(np.sqrt(n)))),
        ('n/2 dirs', lambda n: max(1, n // 2)),
    ]:
        onion_losses = [onion_packing_mse(n, n_dir_func(n), S=S) for n in n_range]
        ax.plot(n_range, onion_losses, '--', linewidth=1.5, label=f'Onion: {n_dir_label}', alpha=0.8)
    
    ax.plot(n_range, pure_angular, 'ko-', linewidth=2, markersize=4, label='Pure angular')
    
    ax.set_xlabel('n (features)')
    ax.set_ylabel('Expected MSE')
    ax.set_title(f'Angular vs Onion Packing (m=2, S={S})')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')

plt.tight_layout()
plt.show()

print("Note: onion model is approximate — the quantization and co-activation")
print("terms are rough estimates. The key insight is qualitative:")
print("  - At low n: angular packing wins (no quantization cost)")
print("  - At high n: onion becomes competitive (interference grows faster than quantization)")
print("  - Higher sparsity S favors onion (co-activation penalty shrinks)")

## Part 3: Detailed Onion Geometry — How Speed Modulation Works

Let's build an explicit toy onion encoder/decoder and see how it reconstructs features.

In [ ]:
class ExplicitOnionEncoder:
    """Hand-crafted onion encoding for visualization.
    
    All features share one direction. Each feature i maps to a trajectory
    that saturates at shell_radius[i]. The saturation creates speed modulation:
    the feature 'zooms past' inner shells and 'parks' at its assigned radius.
    """
    
    def __init__(self, n_features, direction=None, steepness=10):
        self.n = n_features
        self.direction = direction if direction is not None else np.array([1.0, 0.0])
        self.direction = self.direction / np.linalg.norm(self.direction)
        self.steepness = steepness
        
        # Assign shells: feature 0 (most important) → outermost shell
        # Radii decrease geometrically
        self.shell_radii = np.array([3.0 * (0.65 ** i) for i in range(n_features)])
        
        # Shell boundaries (midpoints between adjacent shell centers)
        self.boundaries = np.zeros(n_features + 1)
        self.boundaries[0] = 0.0
        for i in range(n_features - 1):
            self.boundaries[i + 1] = (self.shell_radii[i] + self.shell_radii[i + 1]) / 2
        self.boundaries[-1] = self.shell_radii[0] * 1.5  # Upper boundary
    
    def encode(self, feature_idx, t):
        """Encode: feature i at magnitude t → latent vector."""
        r = self.shell_radii[feature_idx] * sigmoid(t, 0.3, self.steepness)
        return r * self.direction
    
    def decode(self, z):
        """Decode: latent vector → (feature_idx, magnitude)."""
        r = np.dot(z, self.direction)  # Project onto shared direction
        
        if r <= 0:
            return -1, 0.0
        
        # Find which shell r falls in (search from outermost = feature 0)
        for i in range(self.n):
            lo = self.boundaries[i + 1] if i < self.n - 1 else 0
            hi = self.boundaries[i] if i > 0 else self.boundaries[-1]
            
            # Shells go: [boundary_n, boundary_{n-1}], ..., [boundary_1, boundary_0=upper]
            # Actually let's just find the closest shell
        
        # Simpler: assign to nearest shell
        distances = np.abs(r - self.shell_radii)
        feature_idx = np.argmin(distances)
        
        # Reconstruct magnitude: invert the encoding function
        # r = shell_r * sigmoid(t, 0.3, steepness)
        # sigmoid^-1(r/shell_r) = t
        ratio = np.clip(r / self.shell_radii[feature_idx], 1e-6, 1 - 1e-6)
        t_reconstructed = 0.3 + np.log(ratio / (1 - ratio)) / self.steepness
        t_reconstructed = np.clip(t_reconstructed, 0, 2)
        
        return feature_idx, t_reconstructed


# Build and visualize
onion = ExplicitOnionEncoder(4, direction=np.array([1.0, 0.5]))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
t_vals = np.linspace(0.01, 1.5, 300)
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']

# (a) Trajectories in 2D
ax = axes[0]
for i in range(4):
    traj = np.array([onion.encode(i, t) for t in t_vals])
    ax.plot(traj[:, 0], traj[:, 1], color=colors[i], linewidth=2.5,
           label=f'Feature {i} (shell r={onion.shell_radii[i]:.2f})')
    # Mark t=0.5 and t=1.0
    for t_mark in [0.5, 1.0]:
        pt = onion.encode(i, t_mark)
        ax.scatter(*pt, color=colors[i], s=60, zorder=5, edgecolors='black', linewidths=1)

# Draw shells
for i in range(4):
    circle = plt.Circle((0, 0), onion.shell_radii[i], fill=False,
                        color=colors[i], linestyle=':', alpha=0.4)
    ax.add_patch(circle)

ax.set_xlabel('z₀'); ax.set_ylabel('z₁')
ax.set_title('(a) Onion Trajectories in 2D\n(dots at t=0.5, 1.0)')
ax.legend(fontsize=8)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.5, 3.5)
ax.set_ylim(-0.5, 2.0)

# (b) Radius vs magnitude — the speed modulation
ax = axes[1]
for i in range(4):
    traj = np.array([onion.encode(i, t) for t in t_vals])
    radii = np.linalg.norm(traj, axis=1)
    ax.plot(t_vals, radii, color=colors[i], linewidth=2.5, label=f'Feature {i}')
    
    # Show shell region
    ax.axhline(onion.shell_radii[i], color=colors[i], linestyle=':', alpha=0.4)

# Comparison: linear trajectory (bias-free)
ax.plot(t_vals, t_vals * onion.shell_radii[0], 'k--', alpha=0.4,
       label='Linear (bias-free)', linewidth=1)

ax.set_xlabel('Input magnitude t')
ax.set_ylabel('||z(t·eᵢ)||')
ax.set_title('(b) Radius vs Magnitude\nSigmoid saturation → speed modulation')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (c) Decode accuracy: encode each feature at various magnitudes, decode, check
ax = axes[2]
t_test = np.linspace(0.05, 1.5, 100)

for i in range(4):
    correct = []
    mag_errors = []
    for t in t_test:
        z = onion.encode(i, t)
        decoded_feat, decoded_t = onion.decode(z)
        correct.append(decoded_feat == i)
        if decoded_feat == i:
            mag_errors.append(abs(t - decoded_t))
        else:
            mag_errors.append(t)  # Total loss if wrong feature
    
    ax.plot(t_test, mag_errors, color=colors[i], linewidth=2, label=f'Feature {i}')

ax.set_xlabel('True input magnitude t')
ax.set_ylabel('|t - t_reconstructed|')
ax.set_title('(c) Reconstruction Error per Feature\n(lower = better)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 4: Comparison — When Does Each Strategy Win?

Let's systematically compare three strategies:
1. **Pure angular**: n distinct directions, linear decoder
2. **Pure onion**: 1 direction, n radial shells, nonlinear decoder
3. **Hybrid**: $\sqrt{n}$ directions, each with $\sqrt{n}$ shells

Across the key parameters: n (feature count), S (sparsity), importance hierarchy.

In [ ]:
def simulate_angular_mse(n, S=0.95, n_trials=10000):
    """Monte Carlo estimate of MSE for angular packing (m=2).
    
    Equal-magnitude, uniformly spaced in [0, pi].
    Linear tied-weight decoder with ReLU.
    """
    if n == 1:
        return 0.0
    
    angles = optimal_ngon_angles(n, sector=np.pi)
    W = np.stack([np.cos(angles), np.sin(angles)], axis=1)  # (n, 2)
    
    total_mse = 0.0
    for _ in range(n_trials):
        # Generate sparse input
        mask = np.random.random(n) > S
        x = mask * np.random.random(n)
        
        # Encode
        z = x @ W  # (2,)
        
        # Decode (tied weights + ReLU)
        x_hat = np.maximum(0, z @ W.T)  # (n,)
        
        total_mse += np.mean((x - x_hat) ** 2)
    
    return total_mse / n_trials


def simulate_onion_mse(n, n_directions=1, S=0.95, steepness=10, n_trials=10000):
    """Monte Carlo estimate of MSE for onion packing (m=2).
    
    Features are distributed across n_directions directions,
    with ceil(n/n_directions) shells per direction.
    Nonlinear (sigmoid) encoder, nearest-shell decoder.
    """
    if n_directions > n:
        n_directions = n
    
    # Assign features to directions
    features_per_dir = [[] for _ in range(n_directions)]
    for i in range(n):
        features_per_dir[i % n_directions].append(i)
    
    # Direction angles (uniformly spaced)
    if n_directions == 1:
        dir_angles = np.array([np.pi / 4])  # 45°
    else:
        dir_angles = np.linspace(0, np.pi, n_directions, endpoint=True)
    
    directions = np.stack([np.cos(dir_angles), np.sin(dir_angles)], axis=1)  # (n_dir, 2)
    
    # Shell radii for each feature
    max_radius = 3.0
    shell_radii = np.zeros(n)
    for d in range(n_directions):
        k = len(features_per_dir[d])
        for local_idx, global_idx in enumerate(features_per_dir[d]):
            shell_radii[global_idx] = max_radius * (0.6 ** local_idx)
    
    # Feature-to-direction mapping
    feat_to_dir = np.zeros(n, dtype=int)
    for d in range(n_directions):
        for idx in features_per_dir[d]:
            feat_to_dir[idx] = d
    
    total_mse = 0.0
    for _ in range(n_trials):
        # Generate sparse input
        mask = np.random.random(n) > S
        x = mask * np.random.random(n)
        
        # Encode: sum sigmoid trajectories along assigned directions
        z = np.zeros(2)
        for i in range(n):
            if x[i] > 0:
                r = shell_radii[i] * sigmoid(x[i], 0.3, steepness)
                z += r * directions[feat_to_dir[i]]
        
        # Decode: for each direction, project z and find nearest shell
        x_hat = np.zeros(n)
        for d in range(n_directions):
            proj = np.dot(z, directions[d])
            if proj <= 0:
                continue
            
            # Find nearest shell on this direction
            feats_on_dir = features_per_dir[d]
            radii_on_dir = shell_radii[feats_on_dir]
            distances = np.abs(proj - radii_on_dir)
            best_local = np.argmin(distances)
            best_feat = feats_on_dir[best_local]
            
            # Reconstruct magnitude
            ratio = np.clip(proj / shell_radii[best_feat], 1e-6, 1 - 1e-6)
            t_hat = 0.3 + np.log(ratio / (1 - ratio)) / steepness
            x_hat[best_feat] = max(0, t_hat)
        
        total_mse += np.mean((x - x_hat) ** 2)
    
    return total_mse / n_trials


# Compare across n for different sparsities
n_range = [2, 3, 4, 5, 6, 8, 10, 12]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for S, ax in zip([0.95, 0.99], axes):
    angular_mses = [simulate_angular_mse(n, S=S, n_trials=5000) for n in n_range]
    onion_1dir = [simulate_onion_mse(n, n_directions=1, S=S, n_trials=5000) for n in n_range]
    onion_2dir = [simulate_onion_mse(n, n_directions=2, S=S, n_trials=5000) for n in n_range]
    onion_sqrt = [simulate_onion_mse(n, n_directions=max(1, int(np.sqrt(n))), S=S, n_trials=5000)
                  for n in n_range]
    
    ax.plot(n_range, angular_mses, 'ko-', linewidth=2, markersize=6, label='Pure angular')
    ax.plot(n_range, onion_1dir, 's--', linewidth=1.5, markersize=5, label='Onion (1 dir)')
    ax.plot(n_range, onion_2dir, '^--', linewidth=1.5, markersize=5, label='Onion (2 dirs)')
    ax.plot(n_range, onion_sqrt, 'D--', linewidth=1.5, markersize=5, label='Onion (√n dirs)')
    
    ax.set_xlabel('n (features)')
    ax.set_ylabel('MSE')
    ax.set_title(f'Strategy Comparison (m=2, S={S})')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 5: The Variable-Gradient Linear Trajectory

Your second idea: what if the trajectory IS a line (positive homogeneity) but the **gradient of the decoder changes along the line**?

With a piecewise-linear decoder (which ReLU networks naturally create), the latent space is partitioned into **polytopes** where the decoder is linear. A feature trajectory (a ray from the origin) can pass through multiple polytopes, and in each one the decoder reads it differently.

This means:
- At **small magnitudes** (inner polytopes), the decoder might map the ray to feature A
- At **large magnitudes** (outer polytopes), the decoder might map the same ray to feature B

This is onion encoding without needing encoder nonlinearity! The encoder can be perfectly linear; it's the **decoder's piecewise-linear structure** that creates the shells.

But there's a catch: with a bias-free encoder, $z = Wx$ is linear, so $z(t \cdot e_i) = t \cdot w_i$. The trajectory is a ray at constant speed. You can't have one feature "speed past" another's region because the speed is uniform. The onion structure comes entirely from the decoder.

With biases in the encoder, you get the "variable speed" effect on top of the decoder's polytope structure.

In [ ]:
def visualize_piecewise_linear_decoder():
    """Show how a ReLU decoder creates polytope regions in latent space.
    
    With decoder: x_hat = ReLU(W_dec @ z + b_dec)
    The ReLU creates different linear regions depending on which neurons are active.
    A ray through latent space can traverse multiple regions.
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Construct a simple 2D → 4-feature decoder
    # W_dec: (4, 2), b_dec: (4,)
    np.random.seed(17)
    
    # Hand-craft decoder weights to create interesting polytope structure
    # Each row of W_dec defines a half-plane where that feature is active
    W_dec = np.array([
        [1.5, 0.5],    # Feature 0: active when z₀ > -b₀/1.5
        [-0.3, 1.8],   # Feature 1: active when z₁ is large
        [0.8, -1.2],   # Feature 2: active when z₀ > z₁ (roughly)
        [-1.0, -0.5],  # Feature 3: active when z is in negative quadrant
    ])
    b_dec = np.array([-0.5, -0.8, -0.3, -0.2])
    
    # Plot decoder regions
    grid_size = 200
    z0 = np.linspace(-2, 3, grid_size)
    z1 = np.linspace(-2, 3, grid_size)
    Z0, Z1 = np.meshgrid(z0, z1)
    Z_grid = np.stack([Z0.ravel(), Z1.ravel()], axis=1)  # (grid_size^2, 2)
    
    # Decode
    X_decoded = np.maximum(0, Z_grid @ W_dec.T + b_dec)  # (grid_size^2, 4)
    
    # (a) Which feature dominates at each point
    ax = axes[0]
    dominant = np.argmax(X_decoded, axis=1).reshape(grid_size, grid_size)
    max_val = np.max(X_decoded, axis=1).reshape(grid_size, grid_size)
    dominant[max_val < 0.01] = -1  # Dead zone
    
    cmap = plt.cm.get_cmap('Set1', 5)
    im = ax.pcolormesh(Z0, Z1, dominant, cmap=cmap, vmin=-1.5, vmax=3.5, shading='auto')
    plt.colorbar(im, ax=ax, ticks=[-1, 0, 1, 2, 3],
                label='Dominant feature (-1=dead)')
    
    # Draw a ray from origin
    ray_dir = np.array([1.0, 0.7])
    ray_dir = ray_dir / np.linalg.norm(ray_dir)
    t_ray = np.linspace(0, 3.5, 100)
    ray_points = np.outer(t_ray, ray_dir)
    ax.plot(ray_points[:, 0], ray_points[:, 1], 'w-', linewidth=3, alpha=0.8)
    ax.plot(ray_points[:, 0], ray_points[:, 1], 'k--', linewidth=1.5)
    ax.scatter(0, 0, color='white', s=100, zorder=10, edgecolors='black')
    
    ax.set_xlabel('z₀'); ax.set_ylabel('z₁')
    ax.set_title('(a) Decoder Feature Map\n(white line = feature trajectory)')
    ax.set_aspect('equal')
    
    # (b) What the decoder outputs along the ray
    ax = axes[1]
    ray_decoded = np.maximum(0, np.outer(t_ray, ray_dir) @ W_dec.T + b_dec)
    feat_colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
    for i in range(4):
        ax.plot(t_ray, ray_decoded[:, i], color=feat_colors[i], linewidth=2,
               label=f'Feature {i}')
    
    # Shade regions where each feature dominates
    dominant_along_ray = np.argmax(ray_decoded, axis=1)
    for i in range(4):
        mask = dominant_along_ray == i
        if mask.any():
            ax.fill_between(t_ray, 0, np.max(ray_decoded, axis=1),
                          where=mask, alpha=0.1, color=feat_colors[i])
    
    ax.set_xlabel('Distance along ray (t)')
    ax.set_ylabel('Decoded feature value')
    ax.set_title('(b) Decoder Output Along the Ray\n(different features dominate at different radii)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # (c) The "gradient" — d(decoded)/dt along the ray
    ax = axes[2]
    for i in range(4):
        gradient = np.gradient(ray_decoded[:, i], t_ray)
        ax.plot(t_ray, gradient, color=feat_colors[i], linewidth=2,
               label=f'Feature {i}')
    
    ax.set_xlabel('Distance along ray (t)')
    ax.set_ylabel('d(feature value)/dt')
    ax.set_title('(c) Decoder Gradient Along Ray\n(piecewise constant = polytope transitions)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='gray', linewidth=0.5)
    
    plt.tight_layout()
    plt.show()
    
    return W_dec, b_dec

W_dec, b_dec = visualize_piecewise_linear_decoder()

### Connection between the two constraints

The two ideas — (1) n-gon packing and (2) onion/variable-speed trajectories — are actually two ends of a spectrum:

| Strategy | Encoder | Decoder | # directions | Features/direction | Best when |
|----------|---------|---------|-------------|-------------------|----------|
| Pure angular (n-gon) | Linear | Linear | n | 1 | Low n/m, any sparsity |
| Hybrid | Linear | Nonlinear | √n | √n | Moderate compression |
| Pure onion | Nonlinear (biased) | Nonlinear | 1 | n | High n/m, high sparsity |

**The transition between these strategies is the nonlinear phase boundary** in the phase diagram!

In [ ]:
# Phase diagram: optimal strategy as a function of n/m and S

n_over_m_range = np.arange(1, 12)
S_range = [0.8, 0.9, 0.95, 0.99]
m = 2

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

for S in S_range:
    crossover_n = None
    angular_wins = []
    
    for n_over_m in n_over_m_range:
        n = n_over_m * m
        
        ang_mse = simulate_angular_mse(n, S=S, n_trials=3000)
        
        # Try several onion configurations, take best
        best_onion = float('inf')
        for n_dir in [1, 2, max(1, int(np.sqrt(n)))]:
            onion_mse = simulate_onion_mse(n, n_directions=n_dir, S=S, n_trials=3000)
            best_onion = min(best_onion, onion_mse)
        
        angular_wins.append(ang_mse < best_onion)
        
        if crossover_n is None and not angular_wins[-1]:
            crossover_n = n_over_m
    
    label = f'S={S}'
    if crossover_n:
        label += f' (crossover at n/m≈{crossover_n})'
    
    ax.plot(n_over_m_range, angular_wins, 'o-', markersize=8, linewidth=2, label=label)

ax.set_xlabel('n/m (compression ratio)', fontsize=12)
ax.set_ylabel('Angular packing wins?', fontsize=12)
ax.set_yticks([0, 1])
ax.set_yticklabels(['Onion/Hybrid better', 'Angular better'])
ax.set_title('When Does Angular vs Onion Packing Win?\n(m=2, Monte Carlo comparison)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey insight: the crossover point depends on sparsity.")
print("Higher sparsity → onion becomes viable at lower n/m,")
print("because co-activation (which kills onion) is rarer.")

## Summary

### Constraint 1: Equal-magnitude features
- Optimal arrangement is regular n-gon (or half-n-gon with non-negative features)
- Interference grows as $\sim n^2$ because minimum angle shrinks as $\pi/(n-1)$
- With importance weighting, optimal arrangement is **not** regular: important features get larger norms and more angular space
- At some $n$, it's better to drop unimportant features than pack them all in

### Constraint 2: Variable-speed linear trajectories (onion)
- Features can share a direction if they occupy different radial shells
- Encoder biases enable sigmoid saturation → speed past other shells, park in own shell
- Even without encoder biases, the **decoder's piecewise-linear structure** (ReLU polytopes) creates natural radial shells
- Onion encoding trades angular interference for radial quantization error
- Onion wins at high compression (large n/m) and high sparsity (rare co-activation)

### Connection to the phase diagram
The transition from linear to nonlinear encoding in the phase diagram corresponds to the point where onion/hybrid packing becomes more efficient than pure angular packing. This transition is modulated by:
- **n/m ratio**: Higher compression pushes toward onion
- **Sparsity S**: Higher sparsity makes onion viable (co-activation penalty shrinks)
- **Depth l**: More decoder layers → finer-grained polytope structure → better radial discrimination
- **Importance hierarchy**: Steeper hierarchy makes partial representation (dropping features) more attractive, delaying the switch to onion